# 03 — Xây dựng đồ thị possession (Graph Construction)

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 3** trong quy trình XAI-Football:
- Chuyển đổi chuỗi đường chuyền trong possession thành đồ thị có hướng không gian - thời gian (spatio-temporal possession graph).
- Đỉnh (Nodes): Cầu thủ tham gia possession, đặc trưng gồm tọa độ trung bình, số lần chuyền/nhận, và vai trò (one-hot role: GK, DF, MF, FW).
- Cạnh (Edges): Từng đường chuyền thực tế, đặc trưng gồm tọa độ bắt đầu/kết thúc, độ dài, góc chuyền (sin, cos), thứ tự thời gian và độ cao chuyền.
- Cắt đồ thị tại thời điểm cú sút đầu tiên (`truncate_at_first_shot: true`) để loại bỏ rò rỉ thông tin sau khi sút.
- Phân tích số lượng đồ thị được tạo và bị loại.
- Điều tra cảnh báo (WARNING) ở kiểm tra `normalized_features_in_[0,1]`.
- Trực quan hóa một đồ thị possession thực tế trên sân cỏ.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 3 — Dựng đồ thị (`scripts/04_build_graphs.py`, `src/xai_football/graphs/`)**.

In [ ]:
import json
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path(".").resolve()
figures_dir = REPO_ROOT / "results" / "figures" / "official"
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 150

## 3. Input và nguồn dữ liệu
- File đồ thị PyG: `data/processed/official/graphs.pt`
- Index đồ thị: `data/processed/official/graph_index.parquet`
- Danh sách đặc trưng: `data/processed/official/feature_names.json`
- Bảng kiểm tra toàn vẹn đồ thị: `results/tables/official/graph_checks.csv`

## 4. Config/script/module sử dụng
- Cấu hình: `configs/graph.yaml`
- Script: `scripts/04_build_graphs.py`
- Module: `src/xai_football/graphs/build.py`, `dataset.py`, `features.py`.

## 5. Các bước xử lý & Kiểm tra dữ liệu đầu vào

In [ ]:
index_path = REPO_ROOT / "data" / "processed" / "official" / "graph_index.parquet"
feature_names_path = REPO_ROOT / "data" / "processed" / "official" / "feature_names.json"
checks_path = REPO_ROOT / "results" / "tables" / "official" / "graph_checks.csv"

for p in [index_path, checks_path]:
    if not p.exists():
        raise FileNotFoundError(f"Thiếu file: {p}. Vui lòng chạy: python scripts/04_build_graphs.py --config configs/data.yaml")

df_index = pd.read_parquet(index_path)
df_checks = pd.read_csv(checks_path)

with open(feature_names_path, encoding="utf-8") as f:
    feat_names = json.load(f)

print(f"Tổng số đồ thị đã tạo: {len(df_index):,}")
print("Đặc trưng Node:", feat_names.get("node_features"))
print("Đặc trưng Edge:", feat_names.get("edge_features"))

## 6. Kết quả và cách diễn giải
### 6.1. Phân bố số lượng Đỉnh (Nodes) và Cạnh (Edges) trong tập đồ thị

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df_index["n_nodes"], bins=range(2, 12), discrete=True, color="#2980b9", ax=axes[0])
axes[0].set_title("Phân bố số lượng Node (cầu thủ) mỗi đồ thị", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Số lượng đỉnh (Nodes)", fontsize=10)
axes[0].set_ylabel("Số lượng đồ thị", fontsize=10)

sns.histplot(df_index["n_edges"], bins=range(2, 25), discrete=True, color="#e67e22", ax=axes[1])
axes[1].set_title("Phân bố số lượng Cạnh (đường chuyền) mỗi đồ thị", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Số lượng cạnh (Edges)", fontsize=10)
axes[1].set_ylabel("Số lượng đồ thị", fontsize=10)

plt.tight_layout()
plt.savefig(figures_dir / "03_graph_node_edge_dist.png", dpi=300)
plt.savefig(figures_dir / "03_graph_node_edge_dist.svg")
plt.show()

### 6.2. Trực quan hóa một Possession Graph thực tế trên sân cỏ (Pitch Visualization)

In [ ]:
# Lấy một đồ thị mẫu từ dataset (ví dụ possession có bàn thắng/cú sút y=1)
sample_pos = df_index[df_index["y"] == 1].iloc[0]
match_id = int(sample_pos["match_id"])
poss_id = int(sample_pos["possession_id"])

# Đọc các đường chuyền thật của possession này từ possession_events.parquet
passes_path = REPO_ROOT / "data" / "interim" / "official" / "possession_events.parquet"
df_passes = pd.read_parquet(passes_path)
sample_passes = df_passes[(df_passes["match_id"] == match_id) & (df_passes["possession_id"] == poss_id)].sort_values("event_index")

fig, ax = plt.subplots(figsize=(10, 6.5))
# Vẽ khung sân bóng đá (120 x 80 yards)
ax.set_xlim(-5, 125)
ax.set_ylim(-5, 85)
pitch = patches.Rectangle((0, 0), 120, 80, linewidth=2, edgecolor="#27ae60", facecolor="#eafaf1")
ax.add_patch(pitch)
# Vòng tròn giữa sân và đường giữa sân
ax.plot([60, 60], [0, 80], color="#27ae60", linewidth=1.5)
circle = patches.Circle((60, 40), 10, linewidth=1.5, edgecolor="#27ae60", facecolor="none")
ax.add_patch(circle)
# Vòng cấm địa
ax.add_patch(patches.Rectangle((0, 18), 18, 44, linewidth=1.5, edgecolor="#27ae60", facecolor="none"))
ax.add_patch(patches.Rectangle((102, 18), 18, 44, linewidth=1.5, edgecolor="#27ae60", facecolor="none"))

# Vẽ các đường chuyền (cạnh)
for idx, (_, row) in enumerate(sample_passes.iterrows()):
    sx, sy = row["start_x"], row["start_y"]
    ex, ey = row["end_x"], row["end_y"]
    ax.annotate(
        "", xy=(ex, ey), xytext=(sx, sy),
        arrowprops=dict(arrowstyle="-|>", color="#e74c3c", lw=2.5, mutation_scale=15)
    )
    ax.text((sx + ex) / 2, (sy + ey) / 2 + 1.5, f"P{idx+1}", color="#c0392b", fontweight="bold", fontsize=9)

# Vẽ vị trí các cầu thủ (đỉnh)
player_locs = sample_passes.groupby("passer_id")[["start_x", "start_y"]].mean()
for pid, row in player_locs.iterrows():
    ax.scatter(row["start_x"], row["start_y"], s=250, color="#2980b9", edgecolor="black", zorder=5)
    ax.text(row["start_x"], row["start_y"] - 3, f"Player {pid}", ha="center", fontsize=8, fontweight="bold", color="#1f618d")

ax.set_title(f"Minh họa Đồ thị Possession thực tế (Match {match_id}, Poss {poss_id}, Label={sample_pos['y']})", fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("Chiều dài sân (Yards - hướng tấn công sang phải)", fontsize=10)
ax.set_ylabel("Chiều rộng sân (Yards)", fontsize=10)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(figures_dir / "03_sample_possession_graph.png", dpi=300)
plt.savefig(figures_dir / "03_sample_possession_graph.svg")
plt.show()

## 7. Kiểm tra chất lượng (Quality Checks) & Điều tra Graph Warning

### 7.1. Bảng kiểm tra toàn vẹn đồ thị

In [ ]:
print("=== GRAPH INTEGRITY CHECKS ===")
display(df_checks)

### 7.2. Điều tra chi tiết cảnh báo `normalized_features_in_[0,1]`

Trong bảng trên, phép kiểm tra `normalized_features_in_[0,1]` có trạng thái **WARNING** với 1 đồ thị ngoài khoảng:
- **Nguyên nhân**: Tại Graph index 13555 (`match_id = 3754087`, `possession_id = 46`), cầu thủ James Milner thực hiện một quả phạt góc (Corner Kick).
- Tọa độ bóng đặt tại chấm phạt góc góc sân: $x = 120.9$ yards, $y = 2.8$ yards. Do vòng cung phạt góc nằm hơi lệch ra ngoài vạch cuối sân 120 yards của sân bóng đá, tọa độ $x$ thực tế là $120.9$.
- Khi chuẩn hóa bằng công thức $x / 120.0$, giá trị đạt $120.9 / 120.0 = 1.0075$.
- **Kết luận**: Đây là hiện tượng **hợp lệ theo miền vật lý bóng đá thực tế** của dữ liệu StatsBomb, không phải lỗi logic phần mềm. Cảnh báo này được giữ nguyên ở mức WARNING có giải thích rõ ràng nhằm bảo toàn tính trung thực của kiểm toán, không hạ mức giả tạo.

## 8. Output được lưu ở đâu
- Dữ liệu đồ thị: `data/processed/official/graphs.pt`, `graph_index.parquet`, `feature_names.json`.
- Bảng kiểm tra: `results/tables/official/graph_checks.csv`.
- Biểu đồ:
  - `results/figures/official/03_graph_node_edge_dist.png` (và `.svg`)
  - `results/figures/official/03_sample_possession_graph.png` (và `.svg`)

## 9. Cách tái chạy
```bash
python scripts/04_build_graphs.py --config configs/data.yaml
```

## 10. Giới hạn và giả định
- Đồ thị chỉ mô hình hóa mạng lưới chuyền bóng của đội tấn công (on-ball passing networks).
- Đồ thị được cắt tại cú sút đầu tiên (`truncate_at_first_shot: true`). Các possession sau khi cắt chỉ còn < 2 đường chuyền bị loại (318 đồ thị bị loại).